# 01 — Data Download

Este notebook descarga curvas de luz reales desde los servidores de la NASA (MAST)
usando `lightkurve` y las guarda en `data/raw/` para su posterior análisis.

### Estrellas incluidas
| Estrella | Planetas | Nota |
|---|---|---|
| Kepler-11 | 6 | Sistema multiplanetario compacto |
| Kepler-22 | 1 | Primer planeta en zona habitable |
| Kepler-452 | 1 | 'Primo' de la Tierra |
| Kepler-186 | 5 | Kepler-186f en zona habitable |
| Kepler-90 | 8 | Sistema con más planetas conocido |
| KIC 3114789 | 0 | Estrella de control |
| KIC 4914423 | 0 | Estrella de control |
| KIC 6278762 | 0 | Estrella de control |
| KIC 7940546 | 0 | Estrella de control |
| KIC 9410930 | 0 | Estrella de control |

> Requiere conexión a internet. La descarga puede tardar varios minutos.

## 0. Imports y configuración

In [ ]:
import os
import sys
sys.path.append('..')

from exoplanets.data.loader import LightCurveLoader

# Crear carpetas si no existen
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

loader = LightCurveLoader(mission='Kepler', cadence='long')

print('Configuración lista')
print(f'data/raw/       → {os.path.abspath("../data/raw")}')
print(f'data/processed/ → {os.path.abspath("../data/processed")}')

## 1. Estrellas CON exoplanetas confirmados

In [ ]:
estrellas_con_planetas = [
    ('Kepler-11',  1),   # 6 planetas — sistema multiplanetario
    ('Kepler-22',  1),   # 1 planeta  — zona habitable
    ('Kepler-452', 1),   # 1 planeta  — análogo a la Tierra
    ('Kepler-186', 1),   # 5 planetas — Kepler-186f zona habitable
    ('Kepler-90',  1),   # 8 planetas — récord de planetas en un sistema
]

descargados = []
fallidos = []

for nombre, quarter in estrellas_con_planetas:
    try:
        print(f' Descargando {nombre} Q{quarter}...', end=' ')
        lc = loader.fetch(nombre, quarter=quarter)
        filename = nombre.replace('-', '_').lower()
        path = f'../data/raw/{filename}_q{quarter}_raw.csv'
        lc.to_csv(path, index=False)
        descargados.append(nombre)
        print(f' {len(lc)} puntos guardados → {filename}_q{quarter}_raw.csv')
    except Exception as e:
        fallidos.append(nombre)
        print(f' Error: {e}')

print(f'\n Resultado: {len(descargados)}/{len(estrellas_con_planetas)} descargadas correctamente')

## 2. Estrellas SIN planetas conocidos (control)

In [ ]:
estrellas_sin_planetas = [
    'KIC 3114789',
    'KIC 4914423',
    'KIC 6278762',
    'KIC 7940546',
    'KIC 9410930',
]

descargados_ctrl = []
fallidos_ctrl = []

for kic in estrellas_sin_planetas:
    try:
        print(f'  Descargando {kic}...', end=' ')
        lc = loader.fetch(kic, quarter=1)
        filename = kic.replace(' ', '_').lower()
        path = f'../data/raw/{filename}_q1_raw.csv'
        lc.to_csv(path, index=False)
        descargados_ctrl.append(kic)
        print(f' {len(lc)} puntos guardados → {filename}_q1_raw.csv')
    except Exception as e:
        fallidos_ctrl.append(kic)
        print(f' Error: {e}')

print(f'\n Resultado: {len(descargados_ctrl)}/{len(estrellas_sin_planetas)} descargadas correctamente')

## 3. Verificar archivos guardados

In [ ]:
import pandas as pd

archivos = sorted(os.listdir('../data/raw'))
print(f' Archivos en data/raw/: {len(archivos)}\n')

resumen = []
for f in archivos:
    path = f'../data/raw/{f}'
    df = pd.read_csv(path)
    tiene_planeta = 'kepler' in f.lower()
    resumen.append({
        'archivo': f,
        'puntos': len(df),
        'con_planeta': 'Yes' if tiene_planeta else 'No'
    })

pd.DataFrame(resumen)

## 4. Resumen final

In [ ]:
total = len(descargados) + len(descargados_ctrl)
print('=' * 45)
print('           RESUMEN DE DESCARGA')
print('=' * 45)
print(f'  Estrellas con planetas : {len(descargados)}')
print(f'  Estrellas de control   : {len(descargados_ctrl)}')
print(f'  Total descargadas      : {total}')
if fallidos or fallidos_ctrl:
    print(f'  Fallidas               : {fallidos + fallidos_ctrl}')
print('=' * 45)
print('\n Datos guardados en data/raw/')
print('  Continúa en 02_data_exploration.ipynb')